# Stefan Problem with Cattaneo Diffusion: Enthalpy Method

$$
H_t + \nabla\cdot q = 0, \qquad \tau q_t + q = -\nabla u, \qquad u = \beta(H).
$$

# Cattaneo Solver

We replace Fourier's law by the Cattaneo law $\tau q_t + q = -\nabla u$, so the heat flux $q$ is now an unknown. As in `EnthalpySolver`, $H$ lives on the cells, and each component $q_a$ lives on the faces normal to the axis $a$. In each step we first update the flux, treating the relaxation term implicitly and $\nabla u$ explicitly, and then we update $H$ with the divergence of the new flux:
$$
q^{n+1} = \frac{\tau q^n - \Delta t\, \nabla_h u^n}{\tau + \Delta t}, \qquad H^{n+1} = H^n - \Delta t\, \nabla_h\cdot q^{n+1} + \Delta t\,\varepsilon\,\Delta_h u^n.
$$
For $\tau = 0$ the first equation gives $q^{n+1} = -\nabla_h u^n$, and since $\nabla_h\cdot\nabla_h = \Delta_h$ we recover exactly the scheme of `EnthalpySolver`.

The last term is a numerical viscosity with $\varepsilon = \kappa h\sqrt{L_\beta/\tau}$ ($\varepsilon = 0$ for $\tau = 0$). For $\tau > 0$ the temperature jumps at the front, and without this term the scheme produces oscillations behind the front that do not decrease when we refine the mesh. Since $\varepsilon = O(h)$, it vanishes as $h \to 0$.

The time step is the largest one allowed by the von Neumann condition of the linearized scheme ($u = L_\beta H$),
$$
(1+\varepsilon)\mu\,\Delta t^2 + 2(\varepsilon\mu\tau - 1)\Delta t - 4\tau < 0, \qquad \mu = \frac{2L_\beta\,\mathrm{diag}}{h^2},
$$
where diag is the same as in `EnthalpySolver` ($2d$ plus one for each axis with a Dirichlet face). For $\tau = 0$ this is $\Delta t < h^2/(L_\beta\,\mathrm{diag})$, the bound of the Fickian solver. Unlike the Fickian case, for $\tau > 0$ the scheme is not monotone.

# Exact Solution (Travelling Wave)

To test the scheme we use a travelling wave with a melting front moving at constant speed $V$, $s(t) = s_0 + Vt$. Ahead of the front ($\xi = x - s(t) > 0$) there is solid at melting temperature, $u = q = H = 0$, and behind it there is liquid. Writing $b = 1 - \tau V^2$ (we need $b > 0$, i.e. $V < 1/\sqrt{\tau}$, the speed of the thermal waves in the liquid), the solution is
$$
u=\frac{1}{\mathrm{St}}\Big(\frac{e^{-V\xi/b}}{b}-1\Big),\quad q=\frac{V}{\mathrm{St}\,b}\,e^{-V\xi/b}\quad(\xi<0);\qquad u=q=H=0\quad(\xi>0),
$$
with $H = u + 1/\mathrm{St}$ in the liquid. For $\tau > 0$ the temperature is discontinuous at the front: on the liquid side $u^- = \tau V^2/(\mathrm{St}\,b)$, as required by the Rankine–Hugoniot conditions. For $\tau = 0$ we get the usual wave, with $u = 0$ at the front.

In the tests we solve on $[0, 1]$ with the exact temperature at $x = 0$, $\theta = 0$ at $x = 1$, and the exact enthalpy and flux ($q = VH$) as initial data.

In [1]:
from enthalpy import *
from cattaneo import CattaneoSolver
from exact_solutions import tw_H, tw_theta, cell_average, tw_solver

# Verification

In [2]:
mat = Material(St=1.0)
V, s0, T = 1.0, 0.1, 0.4                 # the front goes from x = 0.1 to x = 0.5
taus = [0.0, 0.01, 0.1, 0.5]

print(f"{'tau':>5} {'N':>5} {'L1 error H':>11} {'front error':>12} {'energy balance':>15}")
for tau in taus:
    for N in [100, 200, 400, 800]:
        solver = tw_solver(N, mat, V, tau, s0)
        E0 = solver.total_enthalpy()
        solver.run(T)

        H_ex = cell_average(lambda X: tw_H(X - s0 - V * T, mat, V, tau), N, solver.h)
        err = np.abs(solver.H - H_ex).sum() * solver.h
        front = solver.liquid_volume() - (s0 + V * T)
        balance = solver.total_enthalpy() - E0 - solver.boundary_energy
        print(f"{tau:>5} {N:>5} {err:>11.2e} {front:>12.2e} {balance:>15.1e}")
    print()

  tau     N  L1 error H  front error  energy balance
  0.0   100    2.75e-04     9.31e-05        -3.4e-15
  0.0   200    1.05e-04     3.96e-05        -6.8e-15
  0.0   400    4.04e-05     1.61e-05        -2.1e-15
  0.0   800    1.52e-05     6.35e-06        -6.8e-15

 0.01   100    7.92e-03     5.95e-03         3.3e-16
 0.01   200    4.32e-03     2.99e-03         0.0e+00
 0.01   400    2.36e-03     1.50e-03         7.8e-16
 0.01   800    1.26e-03     7.52e-04         2.1e-15

  0.1   100    5.85e-03     6.94e-04         4.4e-16
  0.1   200    3.24e-03     2.75e-04         1.0e-15
  0.1   400    1.61e-03     1.71e-04         1.1e-16
  0.1   800    7.98e-04     8.82e-05         3.3e-16

  0.5   100    1.01e-02     1.89e-03         6.7e-16
  0.5   200    5.64e-03     9.31e-04         8.9e-16
  0.5   400    2.97e-03     4.24e-04         1.3e-15
  0.5   800    1.55e-03     1.71e-04        -2.9e-15



# Visualization

In [3]:
from visualization import *

In [4]:
N = 400
x = (np.arange(N) + 0.5) / N
runs = {}
for tau in taus:
    solver = tw_solver(N, mat, V, tau, s0)
    times, F, TH = record(solver, t_end=T, n_frames=60, power=1.0)       # the front moves with constant speed
    TH_ex = np.array([tw_theta(x - s0 - V * t, mat, V, tau) for t in times])
    runs[tau] = (times, F, TH, TH_ex)

In [ ]:
ani = animate_1d_taus(x, runs)
HTML(ani.to_jshtml())
# ani.save("tw_taus.mp4", fps=15, dpi=110)